(sec_ex_30_2)=

# Exercise 30.2: Adding cooperativity to the Razumova model

In [Exercise 30.1](sec_ex_30_1), we implemented the basic four-state Razumova model with constant rates. We observed that it cannot reproduce the steep, sigmoidal force-pCa curves found experimentally.

In this exercise, we will extend the model by introducing the three cooperative mechanisms described in [the theory](sec_razumova_cooperativity_theory): RU-RU ($u$), XB-XB ($v$), and XB-RU ($w$) cooperativity.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

## 30.2a: Implement the cooperative rate computation

The cooperative model modifies the kinetic rates at each time step based on the current state of the system. We need to compute the rates in the correct order:

1. Compute the **state subpopulations** ($\lambda^{\mathrm{on}}$, $\lambda^{A_2}$).
2. Apply **XB-XB cooperativity** to compute $f$ and $f'$.
3. Apply **RU-RU cooperativity** to compute $k_{\mathrm{on}}^w$ and $k_{\mathrm{off}}^w$.
4. Apply **XB-RU cooperativity** to compute the final $k_{\mathrm{on}}$ and $k_{\mathrm{off}}$.

**Your task:** Fill in the `...` placeholders in the `rhs_cooperative` function below.


In [ ]:
# Model parameters
R_T = 1

# Ca-dependent base rates
k_0_on = 0  # Base k_on (zero Ca)
k_0_off = 100  # Base k_off (zero Ca)
k_Ca_on = 120  # k_on at saturating Ca
k_Ca_off = 50  # k_off at saturating Ca
Ca_50 = k_Ca_off / k_Ca_on  # Half-saturation constant

# XB kinetic rates (base values, before cooperativity)
f_0 = 50  # Attachment rate
f_prime_0 = 400  # Reverse attachment rate
h = 8  # Powerstroke rate
h_prime = 6  # Reverse powerstroke rate
g = 4  # Detachment rate

# Cooperativity parameters
u = 8  # RU-RU cooperativity
v = 2  # XB-XB cooperativity
w = 3  # XB-RU cooperativity

# Calcium concentration (start with a high value)
Ca = Ca_50 * 100

In [ ]:
def rhs_cooperative(t, y, Ca, u, v, w):
    """Right-hand side of the cooperative Razumova model."""
    D, A_1, A_2 = y

    # Step 0: Ca-dependent base rates
    k_u_on = k_0_on + (k_Ca_on - k_0_on) * Ca / (Ca_50 + Ca)
    k_u_off = k_0_off + (k_Ca_off - k_0_off) * Ca / (Ca_50 + Ca)

    # Step 1: State subpopulations
    R_off = R_T - D - A_1 - A_2
    lambda_A2 = ...  # Fill in: fraction in A_2 state
    lambda_on = ...  # Fill in: fraction of active sites

    # Step 2: XB-XB cooperativity (parameter v)
    f = ...  # Fill in: cooperative attachment rate
    f_prime = ...  # Fill in: cooperative reverse attachment rate

    # Step 3: RU-RU cooperativity (parameter u)
    k_w_on = ...  # Fill in: k_on after RU-RU cooperativity
    k_w_off = ...  # Fill in: k_off after RU-RU cooperativity

    # Step 4: XB-RU cooperativity (parameter w)
    k_on = ...  # Fill in: final k_on
    k_off = ...  # Fill in: final k_off

    # ODEs (same structure as the basic model)
    dD_dt = k_on * R_off + f_prime * A_1 + g * A_2 - (k_off + f) * D
    dA1_dt = f * D + h_prime * A_2 - (f_prime + h) * A_1
    dA2_dt = h * A_1 - (h_prime + g) * A_2

    return [dD_dt, dA1_dt, dA2_dt]

In [ ]:
# Solve the cooperative model
t_span = (0, 10)
t_eval = np.linspace(*t_span, 5000)
y0 = [0.01, 0.01, 0.01]

sol = solve_ivp(
    rhs_cooperative,
    t_span,
    y0,
    t_eval=t_eval,
    method="RK45",
    args=(Ca, u, v, w),
)

# Plot state probabilities
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(sol.t, sol.y[0], label=r"$D$")
axes[0].plot(sol.t, sol.y[1], label=r"$A_1$")
axes[0].plot(sol.t, sol.y[2], label=r"$A_2$")
axes[0].set(
    xlabel="Time (s)",
    ylabel="State probability",
    title=f"Cooperative model (u={u}, v={v}, w={w})",
    ylim=(0, 1),
)
axes[0].legend()

# Force development
A_2_sol = sol.y[2]
axes[1].plot(sol.t, A_2_sol, color="C3")
axes[1].set(
    xlabel="Time (s)",
    ylabel="Relative force ($A_2$)",
    title="Force development",
    xlim=(0, 1),
)

plt.show()

## 30.2b: Generate the steady-state Force-pCa curve

The key test for the cooperative model is whether it can reproduce the steep, sigmoidal **force-pCa curve**. To generate this curve:

1. Sweep over a range of calcium concentrations.
2. For each $[\mathrm{Ca}^{2+}]$, solve the ODE system to steady state.
3. Record the steady-state $A_2$ value (proportional to force).
4. Plot force vs. pCa ($= -\log_{10}[\mathrm{Ca}^{2+}]$).

**Your task:** Complete the loop below by filling in:

- The `solve_ivp` call with the correct RHS function and time span (integrate long enough to reach steady state).
- The line that extracts the final (steady-state) $A_2$ value from the solution object.


In [ ]:
# Sweep over calcium concentrations
Ca_values = np.logspace(-2, 2, 30) * Ca_50  # From very low to very high Ca
steady_state_force = []

for Ca_val in Ca_values:
    sol = solve_ivp(
        ___,  # the RHS function
        ___,  # time span (0, T) — long enough for steady state
        [0.01, 0.01, 0.01],  # initial condition [D, A1, A2]
        method="LSODA",  # efficient adaptive stiff/non-stiff solver
        args=(Ca_val, u, v, w),
    )
    # Extract the final (steady-state) A2 value
    steady_state_force.append(___)

steady_state_force = np.array(steady_state_force)
pCa = -np.log10(Ca_values)

# Plot the force-pCa curve
plt.plot(pCa, steady_state_force / steady_state_force.max(), "o-", color="C0")
plt.xlabel("pCa ($-\\log_{10}[\\mathrm{Ca}^{2+}]$)")
plt.ylabel("Normalised force")
plt.title(f"Steady-state Force-pCa curve (u={u}, v={v}, w={w})")
plt.gca().invert_xaxis()
plt.show()


**Questions:**

1. Does the resulting force-pCa curve look sigmoidal? How does its steepness compare to the experimental data shown in the theory chapter?
2. What is the approximate $\text{pCa}_{50}$ (the pCa at half-maximal force) of your curve?


## 30.2c: Interactive cooperativity explorer

Use the interactive widget below to explore how the three cooperativity parameters ($u$, $v$, $w$) affect the force-pCa relationship and force development dynamics.


In [ ]:
import ipywidgets as widgets


def cooperativity_widget(u=8, v=2, w=3):
    """Interactive force-pCa curve with adjustable cooperativity."""
    Ca_values = np.logspace(-2, 2, 30) * Ca_50
    ss_force = []

    for Ca_val in Ca_values:
        sol = solve_ivp(
            rhs_cooperative,
            (0, 50),
            [0.01, 0.01, 0.01],
            method="LSODA",
            args=(Ca_val, u, v, w),
        )
        ss_force.append(sol.y[2, -1])

    ss_force = np.array(ss_force)
    pCa_vals = -np.log10(Ca_values)

    # Also solve at highest Ca for force development
    t_eval_fd = np.linspace(0, 2, 500)
    sol_fd = solve_ivp(
        rhs_cooperative,
        (0, 2),
        [0.01, 0.01, 0.01],
        t_eval=t_eval_fd,
        method="LSODA",
        args=(Ca_values[-1], u, v, w),
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

    # Force-pCa
    f_max = ss_force.max()
    if f_max > 0:
        axes[0].plot(pCa_vals, ss_force / f_max, "o-", color="C0")
    else:
        axes[0].plot(pCa_vals, ss_force, "o-", color="C0")
    axes[0].set(
        xlabel="pCa",
        ylabel="Normalised force",
        title=f"Force-pCa (u={u}, v={v}, w={w})",
        ylim=(-0.05, 1.1),
    )
    axes[0].invert_xaxis()

    # Force development
    axes[1].plot(sol_fd.t, sol_fd.y[2], color="C3")
    axes[1].set(
        xlabel="Time (s)",
        ylabel="Relative force ($A_2$)",
        title="Force development (high Ca)",
    )

    plt.show()


widgets.interact(
    cooperativity_widget,
    u=widgets.FloatSlider(value=8, min=1, max=20, step=0.5, description="u (RU-RU)"),
    v=widgets.FloatSlider(value=2, min=1, max=5, step=0.25, description="v (XB-XB)"),
    w=widgets.FloatSlider(value=3, min=1, max=8, step=0.25, description="w (XB-RU)"),
);

**Observe and reflect:**

- **$u$ (RU-RU):** Controls the _steepness_ of the force-pCa curve. Try setting $u=1$ (no RU-RU cooperativity) — how does the shape compare?
- **$v$ (XB-XB):** Controls the _maximum force_ and rate of development.
- **$w$ (XB-RU):** _Shifts_ the curve along the pCa axis (calcium sensitivity).

**Questions:**

1. What is the minimum value of $u$ needed to produce a steep, sigmoidal curve that resembles the experimental data?
2. How does the $\text{pCa}_{50}$ shift when you increase $w$? Does the muscle become more or less sensitive to calcium?
3. Set all parameters to 1 (no cooperativity). How does the resulting curve compare to experimental cardiac muscle data? What does this tell you about the role of cooperativity?
